# Google Colab 免費雲端運算環境 — 原文章操作與執行結果

**參考來源：** [如何使用 Google Colab 免費算力訓練自己的模型（GeekPlusA／CSDN）](https://blog.csdn.net/qq122716072/article/details/106779195/)

本 Notebook 依照原文章的操作順序整理：Google Drive → Colab → Google Drive 掛載 → GPU/CPU 查看 → 套件管理 → 目錄切換 → GitHub 專案與資料下載。

> **重要說明（執行結果來源）** 這份檔案中附有**實際執行**的輸出，來自建立此 Notebook 時的 Python 環境，**不是 Google Colab 的實際 GPU／Google Drive 授權結果**。需要 Google 帳號授權、GPU 或網路下載的操作，會在程式執行結果中明確標示「尚未執行」或「不適用」。下載到 [Google Colab](https://colab.research.google.com/) 並選「執行階段 → 全部執行」後，才能得到你自己的 Colab 執行結果。

> 原文第 6 節「訓練自己的模型」標示**未完待續**，因此本檔沒有額外加入 MNIST 或虛構任何模型訓練成果。

## 本檔章節

1. 登入 Google Drive、建立 Google Colab
2. 基本 Python 測試與執行環境
3. 掛載 Google Drive、儲存測試檔案
4. 查看 GPU、CPU 與記憶體資訊
5. Notebook 常用操作：Keras、工作目錄、Git、wget
6. 實作狀態總結與參考資料

**執行方法：** 打開此 `.ipynb` → 若在 Colab，先確認是否需要 GPU → 依序按執行按鈕。下載與安裝步驟為避免額外操作，預設不主動執行，可自行切換開關。

## 一、登入 Google Drive 與建立 Colab 筆記本

原文章先登入 [Google Drive](https://drive.google.com/drive/my-drive)，透過應用程式建立 Colaboratory。現在更方便的方法是直接開啟 [Google Colab](https://colab.research.google.com/) →「新增筆記本」。

以下程式用於確認 Notebook 可以執行 Python。

In [1]:
print("Hello Google Colab")
print("Google Colab 環境教學：程式執行成功")

Hello Google Colab
Google Colab 環境教學：程式執行成功


## 二、確認目前的 Python 與執行環境

執行結果會顯示程式當下的環境：若預覽輸出顯示「非 Colab」，代表它是製作 Notebook 時使用的其他 Python 執行環境，**不能當作 Google Colab 的硬體規格**。

In [2]:
import os
import sys
import platform
import importlib.util

IN_COLAB = "google.colab" in sys.modules or importlib.util.find_spec("google.colab") is not None

print("Python 版本：", sys.version.split()[0])
print("作業系統：", platform.system(), platform.release())
print("機器架構：", platform.machine())
print("目前工作目錄：", os.getcwd())
print("是否在 Google Colab：", "是" if IN_COLAB else "否（此輸出為一般 Python 環境）")

Python 版本： 3.13.5
作業系統： Linux 6.18.55
機器架構： x86_64
目前工作目錄： /mnt/data
是否在 Google Colab： 否（此輸出為一般 Python 環境）


## 三、連接 Google Drive（原文第 3 步）

原文章使用 `google-drive-ocamlfuse`、PPA 和舊版 OAuth 驗證方式；這些步驟不再適合現行 Colab。這裡改用 Colab 官方內建的 `drive.mount()`。

**執行結果說明：** 在 Colab 執行時會出現登入／授權視窗；在一般 Python 環境中會顯示尚未執行，並不會偽造 `Mounted at /content/drive`。

In [3]:
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive")
DRIVE_MOUNTED = False

if IN_COLAB:
    from google.colab import drive
    print("請依 Colab 畫面操作，授權掛載 Google Drive。")
    try:
        drive.mount("/content/drive")
        DRIVE_MOUNTED = DRIVE_ROOT.exists()
        print("Google Drive 掛載狀態：", "成功" if DRIVE_MOUNTED else "尚未成功")
    except Exception as exc:
        print("Google Drive 掛載未完成：", type(exc).__name__, str(exc)[:180])
else:
    print("【尚未執行】需要在 Google Colab 登入並授權 Google Drive。")

【尚未執行】需要在 Google Colab 登入並授權 Google Drive。


### 3.1 查看雲端硬碟檔案並新增測試檔

這裡建立 `MyDrive/Colab_原網站實作` 資料夾與小型 `test.txt` 測試檔。你可以在 Google Drive 中看到真實建立的檔案。

In [4]:
if DRIVE_MOUNTED:
    target = DRIVE_ROOT / "Colab_原網站實作"
    target.mkdir(parents=True, exist_ok=True)
    test_file = target / "test.txt"
    test_file.write_text("Google Colab 與 Google Drive 連線測試成功\n", encoding="utf-8")
    print("資料夾：", target)
    print("新增檔案：", test_file.name)
    print("重新讀取：", test_file.read_text(encoding="utf-8").strip())
    print("此資料夾檔案：", [p.name for p in target.iterdir()][:10])
else:
    print("【尚未執行】Google Drive 未掛載；不會在本機冒充建立雲端檔案。")

【尚未執行】Google Drive 未掛載；不會在本機冒充建立雲端檔案。


## 四、查看 GPU、CPU 與記憶體資訊（原文第 4、5 步）

1. Colab 選單：**執行階段 → 變更執行階段類型 → 硬體加速器 → GPU**。
2. TensorFlow 確認 GPU；有資源時通常能找到像 `/device:GPU:0` 的裝置名稱。
3. `nvidia-smi` 查看 NVIDIA GPU 狀態；無 GPU 時會顯示提示。

**注意：** 免費 Colab 不保證每次都分配 GPU。

In [5]:
try:
    import tensorflow as tf
    print("TensorFlow 版本：", tf.__version__)
    gpu_list = tf.config.list_physical_devices("GPU")
    print("TensorFlow GPU 裝置：", gpu_list)
    print("tf.test.gpu_device_name()：", tf.test.gpu_device_name() or "（未偵測到）")
    print("是否找到 GPU：", bool(gpu_list))
except ImportError:
    print("【未檢查 GPU】此 Python 環境未安裝 TensorFlow；Colab 通常已預裝，請在 Colab 執行。")
except Exception as exc:
    print("【TensorFlow 檢查失敗】", type(exc).__name__, str(exc)[:160])

【未檢查 GPU】此 Python 環境未安裝 TensorFlow；Colab 通常已預裝，請在 Colab 執行。


### 4.1 使用 NVIDIA-SMI 顯示 GPU 資訊

原文指令為 `!/opt/bin/nvidia-smi`，現在通常使用 `!nvidia-smi`。以下是可自動判斷工具是否存在的跨環境版本。

In [6]:
import shutil
import subprocess

nvidia_smi = shutil.which("nvidia-smi")
if nvidia_smi:
    proc = subprocess.run([nvidia_smi], capture_output=True, text=True, timeout=20)
    print(proc.stdout if proc.returncode == 0 else (proc.stderr or "nvidia-smi 無法取得 GPU 資訊"))
else:
    print("【無 NVIDIA-SMI】此環境沒有 nvidia-smi，無法顯示 NVIDIA GPU 資訊。")

【無 NVIDIA-SMI】此環境沒有 nvidia-smi，無法顯示 NVIDIA GPU 資訊。


### 4.2 查看 CPU 與記憶體（原文 `cat /proc/cpuinfo`、`cat /proc/meminfo`）

下面會列出系統真實資料，但**附帶的預覽輸出是本檔建立環境的結果，並非 Google Colab**。到 Colab 重跑會更新。

In [7]:
from pathlib import Path

print("系統：", platform.platform())
print("Python 偵測到的 CPU 數：", os.cpu_count())

cpuinfo = Path("/proc/cpuinfo")
if cpuinfo.exists():
    lines = cpuinfo.read_text(errors="replace").splitlines()
    cpu_model = next((line.split(":", 1)[1].strip() for line in lines if line.lower().startswith("model name")), "無資料")
    print("CPU 型號：", cpu_model)
else:
    print("/proc/cpuinfo：此系統不提供")

meminfo = Path("/proc/meminfo")
if meminfo.exists():
    lines = meminfo.read_text(errors="replace").splitlines()
    wanted = ("MemTotal:", "MemFree:", "MemAvailable:", "SwapTotal:")
    for line in lines:
        if line.startswith(wanted):
            print(line)
else:
    print("/proc/meminfo：此系統不提供")

系統： Linux-6.18.55-x86_64-with-glibc2.41
Python 偵測到的 CPU 數： 5
CPU 型號： AMD EPYC 9V74 80-Core Processor
MemTotal:        6090680 kB
MemFree:         4774396 kB
MemAvailable:    4669580 kB
SwapTotal:             0 kB


### 4.3 磁碟使用情形（補充）

原文重點是 GPU/CPU 與記憶體，此處加上精簡磁碟顯示，方便確認資料下載空間。

In [8]:
import shutil
usage = shutil.disk_usage(os.getcwd())
unit = 1024**3
print(f"目前目錄所在磁碟：總容量 {usage.total/unit:.2f} GiB")
print(f"已使用 {usage.used/unit:.2f} GiB，剩餘 {usage.free/unit:.2f} GiB")

目前目錄所在磁碟：總容量 31.45 GiB
已使用 0.01 GiB，剩餘 29.83 GiB


## 五、Colab Notebook 常用操作（原文第 5 步）

### 5.1 安裝 Keras（`pip install`）

原文示範 `!pip install -q keras`。目前 Colab 通常已包含 Keras，不一定需要重裝。先執行套件版本檢查；如確實需要安裝，將下一段程式的 `INSTALL_KERAS` 改為 `True`。

> 原文也示範 `!pip uninstall -y keras`，但移除可能破壞 TensorFlow/Keras 環境，所以**本檔不會自動卸載套件**。

In [9]:
from importlib import metadata

for package in ("keras", "tensorflow", "pip"):
    try:
        print(f"{package}：{metadata.version(package)}")
    except metadata.PackageNotFoundError:
        print(f"{package}：尚未安裝")

keras：3.13.2
tensorflow：尚未安裝
pip：25.1.1


In [10]:
# 在 Google Colab 需要安裝 Keras 時，再改成 True。
INSTALL_KERAS = False

if INSTALL_KERAS:
    import subprocess, sys
    result = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "keras"], check=False)
    print("pip 安裝返回代碼：", result.returncode)
else:
    print("【已略過】本次未重新安裝 Keras；若需要，請將 INSTALL_KERAS 改為 True。")

【已略過】本次未重新安裝 Keras；若需要，請將 INSTALL_KERAS 改為 True。


### 5.2 查看與切換工作目錄

原文介紹 `!pwd`、`os.chdir('../')`、`%cd /`。以下實際示範 Python 方式的工作目錄切換，執行後再切回原目錄，避免影響後續下載。

In [11]:
import os
from pathlib import Path

original_dir = Path.cwd()
print("切換前：", original_dir)

os.chdir(original_dir.parent)
print("切換到上層：", Path.cwd())

os.chdir(original_dir)
print("切回原目錄：", Path.cwd())
print("目前前 8 個檔案或資料夾：", sorted(p.name for p in Path.cwd().iterdir())[:8])

切換前： /mnt/data
切換到上層： /mnt
切回原目錄： /mnt/data
目前前 8 個檔案或資料夾： ['create_colab_article.py', '人工智慧期中_GitHub版型_完整專案', '人工智慧期中_GitHub版型_完整專案.zip']


### 5.3 下載 GitHub 專案（`git clone`）

原文使用：

```bash
git clone https://github.com/wxs/keras-mnist-tutorial.git
```

**此處只示範下載指令，不訓練其中的 MNIST 模型。** 若要在 Colab 真的下載，將 `DOWNLOAD_GITHUB = True`。

In [12]:
from pathlib import Path
import subprocess

DOWNLOAD_GITHUB = False  # 在 Colab 想下載時改為 True
repo_url = "https://github.com/wxs/keras-mnist-tutorial.git"
repo_parent = Path("/content") if IN_COLAB else Path.cwd()

if DOWNLOAD_GITHUB:
    result = subprocess.run(["git", "clone", repo_url], cwd=str(repo_parent), text=True, capture_output=True)
    print(result.stdout or result.stderr or "指令已執行")
    print("git clone 返回代碼：", result.returncode)
else:
    print("【已略過】GitHub 下載尚未執行")
    print("原文指令：git clone", repo_url)
    print("目標工作目錄：", repo_parent)

【已略過】GitHub 下載尚未執行
原文指令：git clone https://github.com/wxs/keras-mnist-tutorial.git
目標工作目錄： /mnt/data


### 5.4 使用 wget 下載 CSV 資料

原文章示範從 GitHub 的 Rdatasets 儲存庫下載 `Titanic.csv`。

下載功能需要可連接網際網路。將 `DOWNLOAD_CSV = True` 才會下載；未啟用的輸出會清楚標註為略過。

In [13]:
from pathlib import Path
import subprocess
import sys

DOWNLOAD_CSV = False  # 在 Colab 想下載時改為 True
csv_url = "https://raw.githubusercontent.com/vincentarelbundock/Rdatasets/master/csv/datasets/Titanic.csv"
csv_folder = Path("/content/data") if IN_COLAB else (Path.cwd() / "data")

if DOWNLOAD_CSV:
    csv_folder.mkdir(parents=True, exist_ok=True)
    csv_path = csv_folder / "Titanic.csv"
    try:
        from urllib.request import urlretrieve
        urlretrieve(csv_url, csv_path)
        print("下載成功：", csv_path)
        print("檔案大小：", csv_path.stat().st_size, "bytes")
        print("CSV 前 3 行：")
        for line in csv_path.read_text(encoding="utf-8").splitlines()[:3]:
            print(line)
    except Exception as exc:
        print("下載失敗：", type(exc).__name__, str(exc)[:200])
else:
    print("【已略過】Titanic.csv 尚未下載")
    print("原文網址：", csv_url)

【已略過】Titanic.csv 尚未下載
原文網址： https://raw.githubusercontent.com/vincentarelbundock/Rdatasets/master/csv/datasets/Titanic.csv


## 六、執行結果整理與備註

| 原網站項目 | 本 Notebook 的處理方式 |
|---|---|
| Google Drive 登入 | 需到 Google Colab 登入 Google 帳號 |
| Google Drive 掛載 | 採用現行 `drive.mount()`；成功後實際新增／讀取測試檔 |
| GPU 偵測 | 透過 TensorFlow 查詢當前運算裝置 |
| GPU 狀態查看 | `nvidia-smi`，有 NVIDIA GPU 才顯示 |
| Keras 套件 | 真正檢查版本；安裝需自行開啟開關 |
| 目錄指令 | 實際執行工作目錄切換 |
| git clone | 有指令與可開啟的下載開關，預設不下載 |
| wget | 有原始 CSV 網址與可開啟的下載開關，預設不下載 |
| CPU、記憶體 | 顯示執行時系統資訊 |
| 訓練自己的模型 | **原文章未完待續，不產生虛構的訓練結果** |

**執行結果判讀：** 檔案中的現有輸出反映 Notebook 製作環境。要交出「Google Colab 真實執行成果」，請在 Colab 開啟此檔、完成授權、必要時選擇 GPU，然後 **全部執行 → 檔案 → 下載 → 下載 .ipynb**。這樣 Notebook 便會包含你在 Colab 的真正輸出結果。

**資料來源／授權：** [GeekPlusA，〈如何使用 Google Colab 免費算力訓練自己的模型〉，CSDN](https://blog.csdn.net/qq122716072/article/details/106779195/)；原文標示 CC BY-SA 4.0。本檔為繁體中文重新編寫的實作紀錄，保留原文歸屬及連結。